# 第2回：データ処理基礎と記述統計 〜 データの「要約」と「報告」の作法 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session02_data_processing_descriptive.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】データの「型」と尺度分類 (10分)**
2. **【実習1】Pandas によるデータクリーニングの鉄則 (15分)**
3. **【講義2】記述統計の神髄：分布と使い分け (25分)**
   - 正規分布 $\leftrightarrow$ 平均値 (SD)
   - 非正規分布 $\leftrightarrow$ 中央値 (IQR)
   - 現場の鉄則：SD（個体差） vs 95%CI（推定精度）
4. **【実習2】ヒストグラムで分布を見極め、統計量を選ぶ (15分)**
5. **【実習3】プロレベルの表作成：Table 1 形式の出力 (15分)**
6. **【総合演習】「読み手に嘘をつかない」要約レポート作成 (10分)**

---

## 1. セットアップ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style='whitegrid', context='talk')

print("記述統計・レポーティングエンジン、起動。")

## 2. 【最重要講義】分布、代表値、そして報告のルール

### 2.1 正規分布か、非正規分布か？
医学データにおいて、最も基本的で最も破られやすいルールがこれです。論文の「Table 1（患者背景）」を見る際、ここが守られていないものは信頼性が低いとみなされます。

| 特徴 | **正規分布 (Normal)** | **非正規分布 (Non-normal / Skewed)** |
| :--- | :--- | :--- |
| **形状** | 左右対称なベル型。 | 片方に裾が長い（右裾・左裾）、あるいは外れ値がある。 |
| **代表値** | **平均値 (Mean)** が中心を示す。 | **中央値 (Median)** が中心を示す。 |
| **散布度 (広がり)** | **標準偏差 (SD)** | **四分位範囲 (IQR)** または 最小・最大 (Range) |
| **論文での表記例** | Mean ± SD | Median [IQR] |

---

### 2.2 SD・SE・95%CI の使い分け：何を見せたいか？
ここが統計学で最も誤解されやすいポイントです。**「個人のデータの広がり」**を見せたいのか、**「推定された平均値の精密度」**を見せたいのかを明確に区別しましょう。

1. **標準偏差 (SD: Standard Deviation)**
   - **目的**: データの**「個体差（ばらつき）」**を示す。
   - **用途**: **Table 1（患者背景）**。その集団にどのような人がいるかを示す標準的な指標です。
   - **イメージ**: 「このクラスの生徒はだいたい身長 ±10cm くらいの範囲に散らばっている」。
2. **95% 信頼区間 (95% CI: Confidence Interval)**
   - **目的**: 推定した統計量（平均値など）の**「精密度（確からしさ）」**を示す。
   - **用途**: **推論統計（効果の推定）**。母集団の真の値がどのへんにあるかを示します。
   - **イメージ**: 「日本人の真の平均寿命は、95% の確率でこの 0.2 歳の幅の中にあるはずだ」。

> **⚠️ 注意：Table 1 での 95%CI 使用について**
> 記述統計（背景説明）の表で 95%CI を使うのは、一般に**適切ではありません**。95%CI はサンプル数 ($n$) が増えるほど極端に狭くなるため、**「データの実際のバラツキ」を隠して見栄えを良くしている**と誤解（あるいは批判）されるリスクがあります。
> 
> **💡 結論**：
> - 背景を説明するなら **Mean ± SD** または **Median [IQR]** 。
> - 推測した効果や真の平均の幅を論じるなら **[95% CI]** 。
> - 非正規分布に対して「平均値の 95%CI」を出すのは、中心がズレているため誤解を招く禁じ手です。

### 【実践】分布の可視化と手法の選択

In [ ]:
np.random.seed(0)
n = 200

# 1. 正規分布に近いデータ（身長）
height = np.random.normal(170, 6, n)

# 2. 非正規分布（所得や検査値によくある対数正規分布）
income = np.random.lognormal(mean=2, sigma=0.8, size=n) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(height, kde=True, ax=ax1, color='skyblue')
ax1.set_title(f"Normal Distribution\nMean: {np.mean(height):.1f}, SD: {np.std(height):.1f}")

sns.histplot(income, kde=True, ax=ax2, color='orange')
ax2.set_title(f"Skewed Distribution\nMean: {np.mean(income):.1f}, Median: {np.median(income):.1f}")

plt.show()

print("右側のグラフを見てください。平均値(Mean)が大きな値に引っ張られ、")
print("データの中心からズレていることがわかります。これが『中央値を使うべき理由』です。")

## 3. 【実習2】記述統計の算出と 95%CI の計算

In [ ]:
from scipy import stats

# 平均値の 95% 信頼区間を求める関数（正規分布を仮定）
def calculate_95ci(data):
    mean = np.mean(data)
    se = stats.sem(data) # 標準誤差
    ci = stats.t.interval(0.95, len(data)-1, loc=mean, scale=se)
    return ci

ci_height = calculate_95ci(height)
print(f"身長の平均値: {np.mean(height):.2f}")
print(f"身長の 95% 信頼区間: [{ci_height[0]:.2f}, {ci_height[1]:.2f}]")
print("\nこれが意味するのは：『この 200 人から計算した平均値は 170.5 だったが、")
print("母集団（日本人全体）の真の平均は 95% の確率でこの範囲に含まれる』ということです。")

## 4. 【実習3】論文さながらの「Table 1」を作成する

医学論文では、連続変数を「年齢(平均/SD)」「バイタル(中央値/IQR)」のように出し分けます。

In [ ]:
# 患者データの要約テーブル作成シミュレーション
summary_table = pd.DataFrame({
    'Variable': ['Height (cm)', 'Income (k Yen)'],
    'Method': ['Mean ± SD', 'Median [IQR]'],
    'Values': [
        f"{np.mean(height):.1f} ± {np.std(height):.1f}",
        f"{np.median(income):.1f} [{np.percentile(income, 25):.1f} - {np.percentile(income, 75):.1f}]"
    ]
})

print("=== Table 1: Patient Characteristics ===")
display(summary_table)

---

## ✏️ 本日の最終演習問題 (The Reporter's Ethics)

**シナリオ**: ある新薬の治験データ（`recovery_days`: 完治までの日数）があります。このデータには「1人だけ完治に1年かかった」という極端な外れ値が含まれています。

### 課題 1: データの罠を見抜け
`recovery_days` の平均値と中央値を計算してください。どちらの方が「典型的な患者の経過」を代表していると言えますか？

### 課題 2: SD か 95%CI か？
あなたが製薬会社の担当者で、「この薬の効果の安定性（個体差の少なさ）」をアピールしたい場合、SD と 95%CI のどちらをグラフに描くのが不適切（情報の隠蔽に近い）でしょうか。理由とともに述べてください。

### 課題 3: グラフの選択
このデータの分布の形を示すのに最適なグラフを作成し、タイトルを「Recovery Days Distribution」として表示してください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)